In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [8]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )
    return response.output_text

In [12]:
question = 'I just discovered this course. Can I still join now?'
answer = llm(question)
print(answer)

To determine if you can still join the course, check the following:

1. **Registration Deadlines:** Look for any registration or enrollment deadlines.
2. **Course Format:** Some courses may allow rolling admissions, while others may have fixed start dates.
3. **Contact the Institution:** Reach out to the course provider for confirmation and any necessary steps to enroll.

If you provide more details about the course, I might be able to offer additional advice!


In [13]:
context='''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.

edit on GitHub
#Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!
edit on GitHub
#Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

edit on GitHub
#I missed the first homework - can I still get a certificate?
Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

edit on GitHub
#Homework: Why does the content keep changing?
This course is being offered for the first time, and things will keep changing until a given module is ready, at which point it shall be announced. Working on the material or homework in advance will be at your own risk, as the final version could be different.

edit on GitHub

'''

In [14]:
prompt = f'''
You are a helpful assistant for the LLM Zoomcamp course. You have access to the following information about the course:
{context}
Question: {question}
Answer:
'''

In [17]:
question='I stumbled upon this course. I am too busy right now but can i join this two weeks from now?'
answer = llm(prompt)
print(answer)

Yes, you can still join the LLM Zoomcamp course. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted. You can start learning immediately, as registration is just to gauge interest and is not required for participation.


### Creating a RAG

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

#### Knowledge Base

In [26]:
import requests

docs_url='https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw=response.json()

In [32]:
courses_raw[0]['path']

'/json/machine-learning-zoomcamp.json'

In [33]:
documents =[]
url_prefix='https://datatalks.club/faq'

for topic in courses_raw:
    link=url_prefix+topic['path']
    response = requests.get(link)
    faqs=response.json()

    documents.extend(faqs)

In [36]:
documents[1123]

{'id': '405de99093',
 'course': 'mlops-zoomcamp',
 'section': 'Module 4: Deployment',
 'question': 'Pipenv: Pipfile.lock was not created along with Pipfile',
 'answer': 'Use the following command to force the creation of `Pipfile.lock`:\n\n```bash\npipenv lock\n```'}

#### Create search_result

In [37]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'], #all the fields that contain text and should be indexed for search
    keyword_fields=['course'] #all the fields that should be treated as keywords for filtering
)

index.fit(documents)

In [46]:
search_results = index.search(query=question,
                              boost_dict={'question': 2, 'answer': 1}, #optional, if you want to boost the importance of certain fields in the search 
                              filter_dict={'course': 'llm-zoomcamp'}, 
                              num_results=5)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'c2903069a0',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?',
  'answer': 'When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.\n\nIf you want to see what your Display name is, click on the "Edit Course Profile" button.\n\n<{IMAGE:image_1}>\n\n- **First field:** This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This

In [57]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

#### Create user_prompt

In [47]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [48]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [49]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [52]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
A: When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

<{IMAGE:image_1}>

- **First field:** This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
- **Second field:** Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. Thi

In [56]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [54]:
#updated LLM function

def llm(instructions, user_prompt):
    messages=[
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]
    response=openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )
    return response.output_text

#### Full RAG

In [59]:
def rag(question):
    search_result=search(question)
    prompt=build_prompt(question, search_result)
    return llm(INSTRUCTIONS, prompt)

In [60]:
question

'I stumbled upon this course. I am too busy right now but can i join this two weeks from now?'

In [61]:
answer=rag(question)
print(answer)

Yes, you can still join the course two weeks from now. However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted.
